In [ ]:
pip install rasterio numpy pandas pyproj

: 

In [ ]:

import argparse
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import xy
from pyproj import Transformer


def band_names(src):
    # BAG files: band 1 = elevation, band 2 = uncertainty
    if src.driver == "BAG":
        return ["depth", "uncertainty"][: src.count]
    return ["depth"] + [f"band{i}" for i in range(2, src.count + 1)]


def info(path):
    with rasterio.open(path) as src:
        print(f"Driver:     {src.driver}")
        print(f"Size:       {src.width} x {src.height} cells, {src.count} band(s)")
        print(f"CRS:        {src.crs}")
        print(f"Cell size:  {src.res}")
        print(f"Bounds:     {src.bounds}")
        print(f"NoData:     {src.nodata}")
        d = src.read(1, masked=True)
        print(f"Depth range: {d.min()} to {d.max()} (mean {d.mean():.2f})")


def to_xyz(path, out):
    with rasterio.open(path) as src:
        names = band_names(src)
        bands = [src.read(i + 1, masked=True) for i in range(src.count)]
        valid = ~np.ma.getmaskarray(bands[0])
        rows, cols = np.nonzero(valid)
        xs, ys = xy(src.transform, rows, cols)  # cell centres
        df = pd.DataFrame({"x": xs, "y": ys})
        for name, b in zip(names, bands):
            df[name] = b.data[rows, cols]
    df.to_csv(out, index=False)
    print(f"Wrote {len(df):,} points to {out}")


def sample_points(path, points_csv, out, lonlat=False):
    pts = pd.read_csv(points_csv)
    with rasterio.open(path) as src:
        x, y = pts["x"].to_numpy(), pts["y"].to_numpy()
        if lonlat:
            t = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
            x, y = t.transform(x, y)
        names = band_names(src)
        vals = np.array(list(src.sample(zip(x, y), masked=True)))
        nodata = src.nodata
        for i, name in enumerate(names):
            col = vals[:, i].astype(float)
            if nodata is not None:
                col[col == nodata] = np.nan
            pts[name] = col
    pts.to_csv(out, index=False)
    print(f"Sampled {len(pts)} points -> {out} "
          f"({pts['depth'].isna().sum()} outside grid / no data)")


if __name__ == "__main__":
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("mode", choices=["info", "xyz", "points"])
    p.add_argument("grid")
    p.add_argument("points_csv", nargs="?")
    p.add_argument("-o", "--out", default="output.csv")
    p.add_argument("--lonlat", action="store_true",
                   help="points are WGS84 lon/lat rather than grid coordinates")
    a = p.parse_args()

    if a.mode == "info":
        info(a.grid)
    elif a.mode == "xyz":
        to_xyz(a.grid, a.out)
    else:
        if not a.points_csv:
            p.error("points mode needs a CSV of x,y points")
        sample_points(a.grid, a.points_csv, a.out, a.lonlat)